# 15. Auto CDC - 変更を当てる、スナップショットを畳む

`14` までで一区切りついた後、`docs/etl_strategy.md` を書く過程で出てきた宿題。

`08` のLDPでは、テーブルを「宣言する」ところまでしか見なかった。
実際には **すでにある行を更新する** 仕組みもある。それが Auto CDC。

2つあり、名前は似ているが **入力が違う**。

| | `create_auto_cdc_flow` | `create_auto_cdc_from_snapshot_flow` |
|---|---|---|
| 入力 | **変更イベント** の列 | **全件スナップショット** の列 |
| 新旧の判断 | `sequence_by` で指定した列 | スナップショットの版 |
| 削除の検出 | `apply_as_deletes` で明示する | **前の版にあって次の版に無い** こと |
| 源泉の例 | トランザクションログ、Kafka、CDF (`03`) | 日次の全件ダンプ、基幹からのエクスポート |

前者は `05` の `mergeInto` のLDP版。
後者は **Jobs側に相当する機能が無い**。やるなら版同士を比較する処理を自分で書くことになる。

このノートブックで確かめること:

1. 変更イベントから「現在の状態」を作る (SCD Type 1)
2. **同じ入力** から「履歴」を作る (SCD Type 2)
3. 全件スナップショットから履歴を組み立てる
4. `deem_date` のような版を持つモデルで、どう使えるか

**前提**: `00_setup` と `08` を終えていること。`05` を読んでいること。


## 準備


In [1]:
import json

from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()
w = WorkspaceClient(profile="free")

In [2]:
CATALOG = "tech_survey"
SCHEMA = "ldp"

LANDING_ORDERS = f"/Volumes/{CATALOG}/ops/landing/15_cdc_orders"
LANDING_STORES = f"/Volumes/{CATALOG}/ops/landing/15_cdc_stores"

PIPELINE_NAME = "15_auto_cdc_ldp"

## 1. データを置く

2種類のデータを用意する。

**変更イベント (注文)** … 1行が1つの操作を表す。

| order_id | status | event_time | op |
|---|---|---|---|
| 1 | placed | 9/1 10:00 | u |
| 2 | placed | 9/1 10:05 | u |
| 3 | placed | 9/1 10:10 | u |
| 1 | shipped | 9/2 09:00 | u |
| 2 | (削除) | 9/2 09:30 | **d** |

**全件スナップショット (店舗マスタ)** … `deem_date` ごとに全件。まず 9/1 の版だけを置く。

| deem_date | 中身 |
|---|---|
| 2026-09-01 | S1 東京店 / S2 大阪店 / S3 札幌店 |

2つ目の版は、1回流した後に置く。**版が増えていく様子** を見たいため。


In [3]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

for path in (LANDING_ORDERS, LANDING_STORES):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass


# 後で2つ目の版を置くので、まとめておく
def put_snapshot(deem_date: str, stores: list[tuple[str, str]]) -> None:
    """
    店舗マスタのスナップショットを1版ぶん書き出す。

    Parameters
    ----------
    deem_date : str
        版を表す日付。
    stores : list of (str, str)
        店舗IDと店舗名の組。
    """
    rows = []
    for store_id, name in stores:
        rows.append({"store_id": store_id, "store_name": name, "deem_date": deem_date})

    dbutils.fs.put(
        f"{LANDING_STORES}/stores_{deem_date}.json",
        "\n".join(json.dumps(r, ensure_ascii=False) for r in rows),
        True,
    )


changes = [
    {"order_id": 1, "status": "placed", "amount": 150000, "event_time": "2026-09-01T10:00:00", "op": "u"},
    {"order_id": 2, "status": "placed", "amount": 40000, "event_time": "2026-09-01T10:05:00", "op": "u"},
    {"order_id": 3, "status": "placed", "amount": 12000, "event_time": "2026-09-01T10:10:00", "op": "u"},
    {"order_id": 1, "status": "shipped", "amount": 150000, "event_time": "2026-09-02T09:00:00", "op": "u"},
    {"order_id": 2, "status": "placed", "amount": 40000, "event_time": "2026-09-02T09:30:00", "op": "d"},
]
dbutils.fs.put(f"{LANDING_ORDERS}/changes.json", "\n".join(json.dumps(r) for r in changes), True)

put_snapshot("2026-09-01", [("S1", "東京店"), ("S2", "大阪店"), ("S3", "札幌店")])

print("変更イベント:", len(changes), "件")
print("スナップショット: 1版")

変更イベント: 5 件
スナップショット: 1版


## 2. ソースを読む

ソースは [`src/pipelines/15_auto_cdc/transformations/`](../../pipelines/15_auto_cdc/transformations) にある。

### 変更イベントから作る2つのテーブル

まず取り込み先のテーブルを **先に作る**。中身は後続の flow が入れる。

```python
dp.create_streaming_table("cdc_orders_current")

dp.create_auto_cdc_flow(
    target="cdc_orders_current",
    source="cdc_orders_changes",
    keys=["order_id"],
    sequence_by="event_time",           # どちらが新しいかの判断
    stored_as_scd_type=1,               # 上書き。最新の1行だけ残す
    apply_as_deletes=expr("op = 'd'"),  # この条件の行は「削除」として扱う
    except_column_list=["op"],          # op は運搬用なので結果に残さない
)
```

`@dp.table` のようなデコレータではなく、**トップレベルの関数呼び出し** で書く点に注意。
テーブルの中身を返す関数を書くのではなく、「この流し込みをしろ」と命令する形になる。

履歴のほうは、**`stored_as_scd_type` 以外すべて同じ**。

```python
dp.create_auto_cdc_flow(
    target="cdc_orders_history",
    source="cdc_orders_changes",   # 同じソース
    keys=["order_id"],
    sequence_by="event_time",
    stored_as_scd_type=2,          # ← ここだけ違う
    apply_as_deletes=expr("op = 'd'"),
    except_column_list=["op"],
)
```

### スナップショットから作るテーブル

`create_auto_cdc_from_snapshot_flow` は **「最新のスナップショットを持つテーブル」** を入力に取る。
なので、その形のビューを1つ挟む。

```python
@dp.materialized_view()
def stores_latest_snapshot():
    snapshots = spark.read.json(LANDING)
    latest = snapshots.select(F.max("deem_date").alias("_latest"))
    return snapshots.join(latest, snapshots["deem_date"] == latest["_latest"]).drop("_latest")


dp.create_streaming_table("cdc_stores_history")

dp.create_auto_cdc_from_snapshot_flow(
    target="cdc_stores_history",
    source="stores_latest_snapshot",
    keys=["store_id"],
    stored_as_scd_type=2,
    track_history_except_column_list=["deem_date"],
)
```

**`sequence_by` が無い** ことに注目。版の順序がそのまま新旧なので、行ごとの判断が要らない。

`track_history_except_column_list=["deem_date"]` は **このモデルでは必須級**。
`deem_date` は版ごとに必ず変わるので、比較対象に入れると
**中身が同じでも「変わった」と判定されて履歴行ができてしまう**。

### 関数形式は動かなかった

`source` には **関数** を渡す書き方もある。
過去の版をまとめて流し込める形で、`deem_date` モデルとは相性がよさそうに見える。

```python
def next_snapshot_and_version(latest_version):
    snapshots = spark.read.json(LANDING)      # ← ここで落ちた
    ...
    return (snapshots.filter(...), next_date)
```

**この環境では失敗した。** 記録として残す。

```
RuntimeError: The original Spark session is being accessed instead of
the per-flow cloned session during parallel analysis.
```

LDPは複数のフローを **並行して解析する**。
そのとき各フローには専用のセッションが割り当てられるが、
渡した関数の中でグローバルの `spark` に触ると、
「元のセッションを使っている」と判定されて弾かれる。

上のビューを挟む形なら、`spark.read` は **通常のフロー関数の中** で呼ばれるので問題にならない。
過去の版をまとめて流し込みたい場合は別の手を考えることになるが、
**日次で1版ずつ届く運用なら、この形で足りる**。


## 3. デプロイして動かす

リポジトリのルートで実行する。

```sh
databricks bundle deploy
```


In [4]:
PIPELINE_ID = next(
    p.pipeline_id for p in w.pipelines.list_pipelines() if p.name.endswith(PIPELINE_NAME)
)

print(f"{w.config.host}/pipelines/{PIPELINE_ID}")

https://dbc-6f498009-072a.cloud.databricks.com/pipelines/a82a66b1-4985-4e39-b1fa-680d6e486f2b


In [5]:
import time

from databricks.sdk.service.pipelines import UpdateInfoState

DONE = (UpdateInfoState.COMPLETED, UpdateInfoState.FAILED, UpdateInfoState.CANCELED)


def run_pipeline(full_refresh: bool = False) -> UpdateInfoState:
    """
    パイプラインの更新を起動し、終わるまで待つ。

    Parameters
    ----------
    full_refresh : bool, default False
        True にすると、すべてのテーブルを最初から作り直す。

    Returns
    -------
    UpdateInfoState
        更新の最終状態。
    """
    update = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh=full_refresh)

    while True:
        state = w.pipelines.get_update(PIPELINE_ID, update.update_id).update.state
        print(state.value)
        if state in DONE:
            return state
        time.sleep(20)

In [6]:
run_pipeline(full_refresh=True)

CREATED
SETTING_UP_TABLES
RUNNING
COMPLETED


<UpdateInfoState.COMPLETED: 'COMPLETED'>

## 4. SCD Type 1 — 現在の状態だけ


In [7]:
display(spark.table(f"{CATALOG}.{SCHEMA}.cdc_orders_current").orderBy("order_id"))

,amount,event_time,order_id,status,_rescued_data
0,150000,2026-09-02T09:00:00,1,shipped,None
1,12000,2026-09-01T10:10:00,3,placed,None


order_id=1 は `shipped` になっているはず。9/1 の `placed` は残っていない。上書きされたから。

order_id=2 は **消えている**。`op = 'd'` の行が来たときに、削除として扱われた。
`apply_as_deletes` を指定しなければ、`status` が更新されるだけで行は残る。

`op` 列も結果には入っていない。`except_column_list` で外したため。
運搬のためだけに付いている列を結果に残さない、というのは実務でよく要る指定になる。

`05` の `mergeInto` と比べると **書く量が減っている**。
`whenMatched` / `whenNotMatched` / `whenNotMatchedBySource` を並べる代わりに、
キーと順序列を宣言するだけで済んでいる。

代わりに **細かい制御はできない**。`05` でやった
「`shipped` のときだけ更新しない」のような条件は書けない。順序で決まる、という型に乗る必要がある。


## 5. SCD Type 2 — 履歴

ソースも条件も同じで、`stored_as_scd_type` だけ変えたテーブルを見る。


In [8]:
display(
    spark.table(f"{CATALOG}.{SCHEMA}.cdc_orders_history").orderBy("order_id", "__START_AT")
)

,amount,event_time,order_id,status,_rescued_data,__START_AT,__END_AT
0,150000,2026-09-01T10:00:00,1,placed,None,2026-09-01T10:00:00,2026-09-02T09:00:00
1,150000,2026-09-02T09:00:00,1,shipped,None,2026-09-02T09:00:00,None
2,40000,2026-09-01T10:05:00,2,placed,None,2026-09-01T10:05:00,2026-09-02T09:30:00
3,12000,2026-09-01T10:10:00,3,placed,None,2026-09-01T10:10:00,None


`__START_AT` と `__END_AT` という列が増えている。**いつからいつまでその状態だったか** を表す。

- **order_id=1** … 2行ある。`placed` の期間と `shipped` の期間
- **order_id=2** … 1行あるが `__END_AT` が入っている。削除された時点で期間が閉じた
- **order_id=3** … 1行で `__END_AT` は `null`。まだ続いている

**現在の状態を取りたいときは `WHERE __END_AT IS NULL` で絞る。**
Type 1 のテーブルと同じ結果になる。

`05` の `4.` で「削除するか、`cancelled` の印を付けるか」を議論した。
**印を付けて履歴を残す考え方を、仕組みとして持っている** のが Type 2 になる。


## 6. スナップショットから履歴を作る

ここが本題。**変更イベントではなく、全件スナップショットしか無い** 場合。

いまは 9/1 の版だけを置いてある。まずその状態を見る。


In [9]:
display(
    spark.table(f"{CATALOG}.{SCHEMA}.cdc_stores_history").orderBy("store_id", "__START_AT")
)

,deem_date,store_id,store_name,__START_AT,__END_AT
0,2026-09-01,S1,東京店,2026-09-16 08:25:26.141,NaT
1,2026-09-01,S2,大阪店,2026-09-16 08:25:26.141,NaT
2,2026-09-01,S3,札幌店,2026-09-16 08:25:26.141,NaT


3店舗が入り、`__END_AT` はすべて `null`。まだ1版しか無いので、当然この形になる。

### 2つ目の版を置く

9/2 の版を置く。**2箇所変わっている。**

| | 9/1 | 9/2 |
|---|---|---|
| S1 | 東京店 | 東京店 (変化なし) |
| S2 | 大阪店 | **大阪本店** (改名) |
| S3 | 札幌店 | **無い** (消滅) |

もう一度流したときに履歴が何行になるか、予想してほしい。


In [10]:
put_snapshot("2026-09-02", [("S1", "東京店"), ("S2", "大阪本店")])

run_pipeline()

CREATED
RUNNING
COMPLETED


<UpdateInfoState.COMPLETED: 'COMPLETED'>

In [11]:
display(
    spark.table(f"{CATALOG}.{SCHEMA}.cdc_stores_history").orderBy("store_id", "__START_AT")
)

,deem_date,store_id,store_name,__START_AT,__END_AT
0,2026-09-02,S1,東京店,2026-09-16 08:25:26.141,NaT
1,2026-09-01,S2,大阪店,2026-09-16 08:25:26.141,2026-09-16 08:26:25.334
2,2026-09-02,S2,大阪本店,2026-09-16 08:26:25.334,NaT
3,2026-09-01,S3,札幌店,2026-09-16 08:25:26.141,2026-09-16 08:26:25.334


**5行入れたのに、履歴は4行** になっているはず。

| store_id | 行数 | 中身 |
|---|---|---|
| S1 | 1 | 2版とも同じなので1行のまま。`__END_AT` は `null` |
| S2 | 2 | 改名されたので期間が2つに分かれた |
| S3 | 1 | 9/2 の版に無いので `__END_AT` が入って閉じた |

3つとも違う結果になっている。それぞれ確認したいことが違う。

**S1 が1行のままなのが、この仕組みの価値。**
毎回全件が入ってくるのに、履歴には **変化したものしか残らない**。
これが「スナップショットを畳む」ということ。版が増えるほど差は開く。

**S3 の削除検出が、変更イベントとの一番の違い。**
`op = 'd'` のような明示的な指示は **どこにも無い**。
9/2 のスナップショットに S3 が含まれていないという事実だけで、削除と判定されている。

基幹システムからの全件ダンプには「消えた」という情報が入ってこない。
**存在しないことを根拠に削除を判定する** 必要があり、それをこの関数がやっている。

### `deem_date` を比較から外した効果

`track_history_except_column_list=["deem_date"]` を外すと、結果が変わる。

`deem_date` は版ごとに必ず違うので、比較対象に入れると
**S1 も「変わった」と判定されて2行できる**。中身は同じなのに。

つまりこの指定が、**「版が変わったこと」と「中身が変わったこと」を区別している**。
スナップショット方式のデータを畳むときは、必ず要る指定になる。


## 7. `deem_date` モデルでどう使うか

`docs/etl_strategy.md` では、`deem_date` ごとの全件スナップショットを
**そのまま保持する** 方針を採っている。この章はその判断を変えるものではない。

| | スナップショットを保持 (現方針) | Type 2 に畳む (この章) |
|---|---|---|
| 容量 | 店舗数 × `deem_date` 数 | 変化した回数ぶん |
| ある時点の参照 | `WHERE deem_date = X` | `__START_AT <= X AND (__END_AT > X OR __END_AT IS NULL)` |
| ファクトとの結合 | `deem_date` で素直に繋がる | 区間の比較が入る |
| 「いつ変わったか」 | 隣接する版の差分を取る | **自明** |

**参照とファクト結合の素直さが効いている間は、保持したままでよい。**

畳む価値が出るのは次の場合。

- **容量が効いてきたとき** … 店舗数 × 日数が現実的な問題になってきた
- **「いつ変わったか」を頻繁に聞かれるとき** … 価格改定の履歴を追う、など

**両方持つ手もある。** スナップショットは Silver に残したまま、
この章の仕組みで履歴テーブルを別に作る。
容量は増えるが、参照の素直さと変化点の追いやすさを両立できる。
試すなら **店舗マスタ1本だけ並べて作ってみる** のが安全になる。


## 考えてみる

- `create_auto_cdc_flow` で `sequence_by` を指定し忘れると、どうなるでしょうか
- スナップショット方式で、**ある版がまるごと届かなかった** らどうなりますか
- `05` の `mergeInto` を使うべき場面は、まだ残っているでしょうか


### 答え

**Q1. `sequence_by` を忘れると**

**そもそも書けません。必須の引数です。**

`05` では「どれを残すか」を自分で決める必要があり、決めないとキー重複でエラーになりました。
Auto CDC はその判断を **引数として必ず書かせる** 設計になっています。

省略できないのは親切な仕様です。ストリームでは順序が前後して届くのが普通で、
判断基準を決めずに動かすと、実行のたびに結果が変わる不安定な処理になります。

**Q2. ある版がまるごと届かなかったら**

**その版は飛ばされ、次の版との比較になります。**

9/2 が届かず 9/1 → 9/3 と処理された場合、
9/2 で起きた変更は「9/3 に起きた」ことになります。**履歴の期間がずれる。**

エラーにはなりません。届かなかったことに気づく仕組みが別に要ります。
`14` で書いた「黙って進むものは見に行く」がここにも当てはまります。

なお、後から 9/2 が届いても **遡って直りません**。
すでに 9/3 まで進んでいれば、より古い版は対象外になります。
直すには `full_refresh` で作り直すことになります。

**Q3. `mergeInto` の出番は残っているか**

**残っています。** Auto CDC が使えるのは **LDPの中だけ** です。

`13` で見たようなジョブのタスクとして書く処理では、`mergeInto` が引き続き必要になります。
また `05` でやった **条件付きの更新** (「`shipped` のときだけ更新しない」など) は、
Auto CDC の型に収まりません。細かい制御が要るなら `mergeInto` を選ぶことになります。

`docs/etl_strategy.md` の対応表で Jobs 側の列が空にならないのは、こういう理由になります。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [12]:
# for t in ("cdc_stores_history", "stores_latest_snapshot", "cdc_orders_history",
#           "cdc_orders_current", "cdc_orders_changes"):
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{t}")
# for path in (LANDING_ORDERS, LANDING_STORES):
#     try:
#         dbutils.fs.rm(path, True)
#     except NotFound:
#         pass